# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. `{'A': 2, 'B': 1}` - строка `"A"` в цикле повторяется два раза, поэтому добавляется +1 два раза к значению ключа `r["A"]`.
2. `[0, 1, 1]` - функция `f(x)` вызывается два раза и каждый раз append добавляет значение 1 к концу списка.
3. `2` - цикл передает разные строки но `lower()` делает двух строк одинаковым и строчным, поэтому словарь принимает одинаковые ключи из цикла.
4. `{'ab': 2, "abc": 3}` - тут цикл передает одинаковую строку два раза, поэтому ключ остается тем же, а `len(k)` считает количество символов в переданным строке.
5.  `5 и None` - потому что функции `g(n)` ничего не возвращает и поэтому `x` будет `None`, а сама функция `g(n)` вызывает только `print(n)` которая выводить `5`.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [1]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [13]:
# База
def to_num(s):
    return float(s.replace(" ", ""))

def clean_city(s):
    cities = {
        'Astana': 'Астана',
        'Almaty': 'Алматы',
        'Shymkent': 'Шымкент',
        'Karaganda': 'Караганда',
        'Atyrau': 'Атырау',
        'Aktobe': 'Актобе',
        'Астана': 'Астана',
        'Алматы': 'Алматы',
        'Шымкент': 'Шымкент',
        'Караганда': 'Караганда',
        'Атырау': 'Атырау',
        'Актобе': 'Актобе'
    }
    return cities.get(s.strip().title())

category_counts = {}       # 2 (общее число заказов)
category_revenue = {}      # 3 (выручка)
category_paid_counts = {}  # 3 (кол-во только оплаченных)
month_counts = {}          # 4 (заказы по месяцам)

for row in rows:
    # Извлекаем и сразу очищаем нужные поля
    city = clean_city(row['city'])
    category = row['category']
    status = row['status']
    amount = to_num(row['amount'])
    month = row['order_date'][:7] # Извлекаем ГГГГ-ММ

    # 2 (подсчет всех категорий)
    category_counts[category] = category_counts.get(category, 0) + 1

    # 3 (только если статус "оплачен")
    if status == 'оплачен':
        category_revenue[category] = category_revenue.get(category, 0.0) + amount
        category_paid_counts[category] = category_paid_counts.get(category, 0) + 1

    # 4 (подсчет заказов по месяцам)
    month_counts[month] = month_counts.get(month, 0) + 1
    

print("2: Количество заказов в каждой категории")
# Сортируем словарь по значениям (количеству) по убыванию
sorted_categories = sorted(category_counts.items(), key=lambda x: x[1], reverse=True)
for cat, count in sorted_categories:
    print(f"{cat}: {count} заказов")
    
print("\n3: Выручка и средний чек оплаченных заказов")
print(f"{'Категория':<15} | {'Выручка':<15} | {'Средний чек':<15}")
print("-" * 45)

# Сортируем по выручке по убыванию
sorted_revenue = sorted(category_revenue.items(), key=lambda x: x[1], reverse=True)
for cat, revenue in sorted_revenue:
    paid_count = category_paid_counts[cat]
    avg_check = revenue / paid_count if paid_count > 0 else 0.0
    # Форматируем вывод чисел с плавающей точкой (.2f)
    print(f"{cat:<15} | {revenue:<15.2f} | {avg_check:<15.2f}")

print("\n4: Количество заказов по месяцам")
# Сортируем по хронологии месяцев
sorted_months = sorted(month_counts.items())
for month, count in sorted_months:
    print(f"{'Сентябрь' if month == '2026-09' else 'Октябрь'}: {count} заказов")

2: Количество заказов в каждой категории
Одежда: 1529 заказов
Электроника: 1185 заказов
Дом: 1105 заказов
Красота: 991 заказов
Книги: 653 заказов

3: Выручка и средний чек оплаченных заказов
Категория       | Выручка         | Средний чек    
---------------------------------------------
Электроника     | 65270240.00     | 61286.61       
Одежда          | 18321080.00     | 13285.77       
Дом             | 10516320.00     | 10654.83       
Красота         | 5507620.00      | 6188.34        
Книги           | 2294100.00      | 3934.99        

4: Количество заказов по месяцам
Сентябрь: 2782 заказов
Октябрь: 2681 заказов


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [31]:
# Стандарт
# 5. Выручка оплаченных по месяцам и прирост Октября к Сентябрю
monthly_revenue = {}

for row in rows:
    month = row['order_date'][:7]
    amount = to_num(row['amount'])
    if status == 'оплачен':
        monthly_revenue[month] = monthly_revenue.get(month, 0.0) + amount

print("Выручка по месяцам:")
for m, rev in sorted(monthly_revenue.items()):
    print(f"{m}: {rev:,.2f}")

# Расчет изменения Октября к Сентябрю
sept_rev = monthly_revenue.get('2026-09', 0.0)
oct_rev = monthly_revenue.get('2026-10', 0.0)

if sept_rev > 0:
    pct_change = ((oct_rev - sept_rev) / sept_rev) * 100
    print(f"\nИзменение выручки (Октябрь к Сентябрю): {pct_change:+.1f}%")
else:
    print("\nНет данных за сентябрь для расчета динамики.")
    
# 6. Доля отменённых заказов по каждому чистому городу
city_total = {}
city_canceled = {}

for row in rows:
    city = clean_city(row['city'])
    city_total[city] = city_total.get(city, 0) + 1
    status = row['status']
    if status == 'отменён':
        city_canceled[city] = city_canceled.get(city, 0) + 1

print(f"{'Город':<15} | {'Доля отмен':<15}")
print("-" * 30)
for city in sorted(city_total.keys()):
    total = city_total[city]
    canceled = city_canceled.get(city, 0)
    share = (canceled / total) * 100
    print(f"{city:<15} | {share:.1f}%")
    
# 7. Топ-5 клиентов по выручке оплаченных заказов
customer_revenue = {}

for row in rows:
    if status == 'оплачен':
        cust_id = row['customer_id'].strip()
        customer_revenue[cust_id] = customer_revenue.get(cust_id, 0.0) + amount

top_5 = sorted(customer_revenue.items(), key=lambda x: x[1], reverse=True)[:5]
print("Топ-5 клиентов:")
for rank, (cust, rev) in enumerate(top_5, 1):
    print(f"{rank}. {cust}: {rev:,.2f}")

Выручка по месяцам:
2026-09: 60,704,470.00
2026-10: 49,178,890.00

Изменение выручки (Октябрь к Сентябрю): -19.0%
Город           | Доля отмен     
------------------------------
Актобе          | 10.3%
Алматы          | 7.9%
Астана          | 8.8%
Атырау          | 7.5%
Караганда       | 8.9%
Шымкент         | 20.4%
Топ-5 клиентов:
1. C01818: 164,010.00
2. C01076: 134,190.00
3. C01947: 134,190.00
4. C01656: 134,190.00
5. C00420: 134,190.00


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж